In [3]:
import pandas as pd
import numpy as np
from pathlib import Path

from sklearn.model_selection import train_test_split

In [4]:
DATA_DIR = Path("../data/processed")

clinical = pd.read_csv(DATA_DIR / "clinical_manifest.csv")

print("Patients:", clinical["PATIENT_ID"].nunique())
print("Samples:", clinical["SAMPLE_ID"].nunique())
print("Stage distribution:")
print(clinical["stage_binary"].value_counts())

def create_patient_split(
    clinical,
    patient_col="PATIENT_ID",
    stage_col="stage_binary",
    random_state=42
):
    """
    Create 70/15/15 train-validation-test splits at the patient level.
    """

    # Confirm each patient has only one stage label
    stage_per_patient = (
        clinical.groupby(patient_col)[stage_col]
        .nunique()
    )

    if (stage_per_patient > 1).any():
        raise ValueError("Some patients have conflicting stage labels.")

    # One row per patient
    patients = (
        clinical[[patient_col, stage_col]]
        .drop_duplicates(subset=patient_col)
        .copy()
    )

    # 70% train, 30% temporary
    train_patients, temp_patients = train_test_split(
        patients,
        test_size=0.30,
        stratify=patients[stage_col],
        random_state=random_state
    )

    # Split remaining 30% equally:
    # 15% validation, 15% test
    val_patients, test_patients = train_test_split(
        temp_patients,
        test_size=0.50,
        stratify=temp_patients[stage_col],
        random_state=random_state
    )

    return train_patients, val_patients, test_patients

Patients: 1967
Samples: 1967
Stage distribution:
stage_binary
1    1024
0     943
Name: count, dtype: int64


In [5]:
def summarize_split(name, df, stage_col="stage_binary"):
    print(f"{name}")
    print(f"Patients: {len(df)}")
    print(df[stage_col].value_counts())
    print(df[stage_col].value_counts(normalize=True).round(3))
    print()

In [6]:
train_patients, val_patients, test_patients = create_patient_split(clinical)

summarize_split("TRAIN", train_patients)
summarize_split("VALIDATION", val_patients)
summarize_split("TEST", test_patients)

TRAIN
Patients: 1376
stage_binary
1    716
0    660
Name: count, dtype: int64
stage_binary
1    0.52
0    0.48
Name: proportion, dtype: float64

VALIDATION
Patients: 295
stage_binary
1    154
0    141
Name: count, dtype: int64
stage_binary
1    0.522
0    0.478
Name: proportion, dtype: float64

TEST
Patients: 296
stage_binary
1    154
0    142
Name: count, dtype: int64
stage_binary
1    0.52
0    0.48
Name: proportion, dtype: float64



In [7]:
train_patients = train_patients.copy()
val_patients = val_patients.copy()
test_patients = test_patients.copy()

train_patients["split"] = "train"
val_patients["split"] = "validation"
test_patients["split"] = "test"

splits = pd.concat(
    [train_patients, val_patients, test_patients],
    ignore_index=True
)

assert len(splits) == 1967
assert splits["PATIENT_ID"].is_unique

splits.to_csv(DATA_DIR / "patient_splits.csv", index=False)

print(splits["split"].value_counts())
print("Split assignments saved.")

split
train         1376
test           296
validation     295
Name: count, dtype: int64
Split assignments saved.


In [8]:
# Verify no patient appears in multiple splits
assert set(train_patients["PATIENT_ID"]).isdisjoint(
    set(val_patients["PATIENT_ID"])
)

assert set(train_patients["PATIENT_ID"]).isdisjoint(
    set(test_patients["PATIENT_ID"])
)

assert set(val_patients["PATIENT_ID"]).isdisjoint(
    set(test_patients["PATIENT_ID"])
)

print("PASS: No patient overlap between splits.")

PASS: No patient overlap between splits.
